# 02 — Tokenize & embed

Fit the key–value–time tokenizer, tokenize the book into shards, and
embed users with a pretrained pragmatiq model, either from shards or
directly from plain Python dicts.

For *how* the tokenizer and model work — the `8·ln(1+Δt/8)` time transform,
TimeRoPE on continuous log-seconds, the profile/event/history encoders, and the
MLM objective — see [`docs/architecture.md`](../docs/architecture.md).

> pragmatiq is an independent implementation inspired by the PRAGMA paper
> (arXiv 2604.08649) and is not affiliated with or endorsed by Revolut.

## Setup

Notebook 02 reuses the synthetic book created by notebook 01.

In [ ]:
from pathlib import Path

from IPython.display import display

from pragmatiq import api
from pragmatiq.data.tokenizer import PragmaTokenizer
from pragmatiq.models.pragmatiq import PragmaModel

WORK = Path("runs_quickstart")
TOKENIZER_CONFIG = {
    "target_vocab": 28000,
    "n_buckets": 64,
}

## Tokenize the book

Tokenization fits the field vocabulary, numeric buckets, text BPE model, and
writes tokenized shards for training and embedding.

In [ ]:
api.tokenize(
    WORK / "raw",
    WORK / "tok",
    config=TOKENIZER_CONFIG,
)

## Inspect the tokenizer

Each field key is classified as numeric (percentile buckets plus a zero bucket),
categorical (one token per value), or textual (byte-level BPE).

> **PRAGMA+Nemotron variant.** To embed high-cardinality text with a frozen text model
> instead of BPE (reconstructed with MSE during pretraining), tokenize with
> `config=configs/data/tokenizer_nemotron.yaml` (or set `text_value_mode="embed"`); the
> subsequent `pretrain` call auto-builds the matching text branch. It is off by default,
> so the BPE path here is unchanged.

In [ ]:
tokenizer = PragmaTokenizer.load(WORK / "tok" / "tokenizer")

print("vocab size:", tokenizer.vocab_size)
display({k: tokenizer.field_kind[k] for k in list(tokenizer.field_kind)[:12]})

## Pretrain a nano model

The nano config is CPU-friendly for a notebook. Use `model_size="small"` and
more steps for a real run, preferably on GPU.

In [ ]:
summary = api.pretrain(
    WORK / "tok",
    "nb02",
    model_size="nano",
    config={"max_steps": 80, "token_budget": 8192},
    runs_root=WORK / "runs",
)

summary["run_dir"]

## Embed records interactively

`PragmaModel.from_pretrained` can embed plain dictionaries, so notebooks do not
need to rebuild parquet shards for every interactive example.

In [ ]:
model = PragmaModel.from_pretrained(summary["run_dir"])

records = [
    {
        "user_id": "demo",
        "events": [
            {
                "ts": 1_700_000_000_000_000,
                "source": "transaction",
                "fields": {
                    "amount": "42.10",
                    "mcc": "5411",
                    "merchant": "TESCO 1",
                },
            }
        ],
        "attributes": {"country": "GB"},
        "lifelong": [],
    }
]

embeddings = model.embed_records(records)
print("embedding shape:", embeddings.shape)